### Colab stuff

In [ ]:
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

import os
os.chdir('/content/drive/MyDrive/ve/variance_analysis')

In [ ]:
!pip install -e /content/drive/MyDrive/ve/variance_analysis

!pip -q install fonttools

!ttx -l fonts/Helvetica.ttc

!sudo mkdir -p /usr/share/fonts/truetype/helvetica
!ttx -y 0 -o /usr/share/fonts/truetype/helvetica/Helvetica-Regular.ttf fonts/Helvetica.ttc
!ttx -y 1 -o /usr/share/fonts/truetype/helvetica/Helvetica-Oblique.ttf fonts/Helvetica.ttc
!ttx -y 2 -o /usr/share/fonts/truetype/helvetica/Helvetica-Bold.ttf fonts/Helvetica.ttc
!ttx -y 3 -o /usr/share/fonts/truetype/helvetica/Helvetica-BoldOblique.ttf fonts/Helvetica.ttc

!fc-cache -fv

import sys, subprocess, importlib.util
from importlib.metadata import version, PackageNotFoundError

# 1) Remove the wrong kaleido
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "kaleido"])

# 2) Install the Plotly kaleido
subprocess.check_call([sys.executable, "-m", "pip", "install", "kaleido==0.2.1"])

# 3) Verify the right module layout
print("Kaleido version:", version("kaleido"))
print("kaleido.scopes.plotly importable?:",
      importlib.util.find_spec("kaleido.scopes.plotly") is not None)


# Explained variance: rodent neurons selective to place or HD

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.io import loadmat
from scipy.io.matlab._mio5_params import mat_struct
import os
from pathlib import Path
import matplotlib.font_manager as fm
import shutil

current_file_path = Path().resolve()


# set up matplotlib formatting too

!mkdir -p ~/.fonts
!cp fonts/Helvetica.ttc ~/.fonts/

cache_dir = os.path.join(os.path.expanduser("~"), ".cache", "matplotlib")
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)

fm._load_fontmanager(try_read_cache=False)
plt.rcParams['font.family'] = 'Helvetica'

import plotly.io as pio
import plotly.graph_objects as go

pio.templates["helvetica"] = go.layout.Template(
    layout=dict(font=dict(family="Helvetica"))
)


In [ ]:
fig, ax = plt.subplots(figsize=(4, 2), dpi=150)
ax.text(0.5, 0.62, 'The quick brown fox', ha='center', va='center', fontsize=16)
ax.axis('off')
plt.show()


## Pfeiffer Place Neurons


In [ ]:
import os

save_path = current_file_path / f'results/rodents/pfeiffer'
data_path = current_file_path / f'data/rodents/pfeiffer'

rats = [1,2,3,4]


RAT_NEURONS = [204, 174, 79, 94]
n_hd_bins = 60
n_xy_bins = 20


In [ ]:
from src.pfeiffer_ratemaps import get_fold_ratemaps

ratemap_paths = []

#rats = [2,3,4]

#RAT_NEURONS = [174, 79, 94]

for rat in rats:
    metadata = get_fold_ratemaps(
        data_path=data_path,
        save_path=save_path,
        rat=rat,
        n_hd_bins=n_hd_bins,
        n_loc_bins=n_xy_bins,
        floor=5
    )
    ratemap_paths.append(metadata['out_npz'])


In [ ]:
from joblib import Parallel, delayed
from src.explained_variance import *
from tqdm import tqdm

rats = list(range(1, 5))
RAT_NEURONS = [204, 174, 79, 94]

sigma_bins = (1, 2)
max_bins = max(n_hd_bins, n_xy_bins**2)
zoom_factor = (max_bins / n_xy_bins**2, max_bins / n_hd_bins)

def process_neuron(n, fold0_maps, fold1_maps, occ_z):
    fold0 = nan_aware_gaussian_filter(fold0_maps[:, :, n], sigma_bins, circular_axes=(1,))
    fold1 = nan_aware_gaussian_filter(fold1_maps[:, :, n], sigma_bins, circular_axes=(1,))
    fold0 = masked_zoom(fold0, zoom_factor)
    fold1 = masked_zoom(fold1, zoom_factor)
    min_occ = np.percentile(occ_z[occ_z > 0], 10)
    ev_xy = 1 - quantify_variance_corr(fold0, fold1, occ_z, axis=0, min_occ=min_occ)[0]
    ev_hd = 1 - quantify_variance_corr(fold0, fold1, occ_z, axis=1, min_occ=min_occ)[0]
    return ev_xy, ev_hd

ev_xy_per_rat = []
ev_hd_per_rat = []

for r in tqdm(rats):
    n_neurons = RAT_NEURONS[r - 1]

    fold0_maps = np.load(save_path / f'Rat{r}/fold0_xyhd_{n_xy_bins}x{n_xy_bins}x{n_hd_bins}.npz')['rate_map']
    fold1_maps = np.load(save_path / f'Rat{r}/fold1_xyhd_{n_xy_bins}x{n_xy_bins}x{n_hd_bins}.npz')['rate_map']
    occ = np.load(save_path / f'Rat{r}/ratemap_xyhd_{n_xy_bins}x{n_xy_bins}x{n_hd_bins}.npz')['occupancy']

    x, y, hd, _ = fold0_maps.shape
    fold0_maps = fold0_maps.reshape(y * x, hd, n_neurons)
    fold1_maps = fold1_maps.reshape(y * x, hd, n_neurons)
    occ = occ.reshape(y * x, hd)

    # Precompute zoomed occupancy once
    occ_z = masked_zoom(occ, zoom_factor)

    # Parallel across neurons
    results = Parallel(n_jobs=-1)(
        delayed(process_neuron)(n, fold0_maps, fold1_maps, occ_z)
        for n in range(n_neurons)
    )

    ev_xy = np.array([r[0] for r in results])
    ev_hd = np.array([r[1] for r in results])

    ev_xy_per_rat.append(ev_xy)
    ev_hd_per_rat.append(ev_hd)

for r in rats:
    np.save(save_path / f'ev_xy_rat_{r}_all_{n_xy_bins}x{n_xy_bins}x{n_hd_bins}-corr.npy', ev_xy_per_rat[r - 1])
    np.save(save_path / f'ev_hd_rat_{r}_all_{n_xy_bins}x{n_xy_bins}x{n_hd_bins}-corr.npy', ev_hd_per_rat[r - 1])
    print(f"Saved EVs for Rat {r}: {len(ev_xy_per_rat[r-1])} neurons")

# the story here is just that the pairwise and the all-told looks basically the same.


In [ ]:
# Load saved EVs (all neurons)
ev_xy_per_rat = []
ev_hd_per_rat = []

for r in rats:
    ev_xy_per_rat.append(np.load(save_path / f'ev_xy_rat_{r}_all_{n_xy_bins}x{n_xy_bins}x{n_hd_bins}-corr.npy'))
    ev_hd_per_rat.append(np.load(save_path / f'ev_hd_rat_{r}_all_{n_xy_bins}x{n_xy_bins}x{n_hd_bins}-corr.npy'))
    print(f"Loaded Rat {r}: {len(ev_xy_per_rat[-1])} neurons")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2)

fig, ax = plt.subplots(figsize=(6, 6))

all_x = np.concatenate(ev_xy_per_rat).astype(float)
all_y = np.concatenate(ev_hd_per_rat).astype(float)
valid = ~np.isnan(all_x) & ~np.isnan(all_y)
xv, yv = all_x[valid], all_y[valid]

for r_idx, r in enumerate(rats):
    x_vals = ev_xy_per_rat[r_idx]
    y_vals = ev_hd_per_rat[r_idx]
    n_neurons = len(x_vals)

    idx = np.arange(n_neurons)
    rng.shuffle(idx)
    idx = idx[:min(30, n_neurons)]

    ax.scatter(x_vals[idx], y_vals[idx], s=16, alpha=0.8, edgecolors='none', color='C0')

lo = 0.0 #float(np.nanmin([xv.min(), yv.min()]))
hi = 1.0 #float(np.nanmax([xv.max(), yv.max()]))
pad = 0.02 * (hi - lo if hi > lo else 1.0)
lo -= pad
hi += pad

ax.plot([lo, hi], [lo, hi], '--', lw=1.0, color='red', label='y = x')
ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)
ax.set_aspect('equal', adjustable='box')
ax.set_xlabel('EV (XY)')
ax.set_ylabel('EV (HD)')
ax.legend(loc='best', frameon=False, fontsize=8)

plt.tight_layout()
plt.savefig(current_file_path / f"figs/pfeiffer_ev_total_best_fit-corr.pdf")
plt.show()


### Take a look at some of the high-scored HD or XY neurons

In [ ]:
import plotly.graph_objects as go
from plotly import io as pio
from src.ann.fig import plot_neuron_2d_3d
from src.explained_variance import nan_aware_gaussian_filter

from tqdm import tqdm

sigma_bins = (1, 1, 4) # 4cm, 4cm, 4 deg

norm_xy = (all_x - lo) / (hi - lo)
norm_hd = (all_y - lo) / (hi - lo)
xy_ratio = norm_xy / (norm_xy + norm_hd)
hd_ratio = norm_hd / (norm_xy + norm_hd)

# neuron_limits tells us which global index belongs to which rat
neuron_limits = np.zeros_like(RAT_NEURONS)
total_neurons = 0
for i in range(len(neuron_limits)):
    neuron_limits[i] = total_neurons
    total_neurons += RAT_NEURONS[i]

most_xy = np.argsort(xy_ratio)[::-1]
most_hd = np.argsort(hd_ratio)[::-1]

for rank_idx in tqdm(range(5)):

    global_idx = most_hd[rank_idx]

    which_rat = np.searchsorted(neuron_limits, global_idx, side="right") - 1
    rat_neuron = global_idx - neuron_limits[which_rat]
    ev = hd_ratio[global_idx]

    map = np.load(
        save_path / f"Rat{which_rat + 1}/ratemap_xyhd_{n_xy_bins}x{n_xy_bins}x{n_hd_bins}.npz"
    )["rate_map"]

    for n in tqdm(range(map.shape[3])):
      map[:, :, :, n] = nan_aware_gaussian_filter(map[:, :, :, n], sigma_bins, circular_axes=(0,0,1)) # whatever

    # cap outliers
    low_p, high_p = 0, 99
    cmin = float(np.nanpercentile(map[:, :, :, rat_neuron], low_p))
    cmax = float(np.nanpercentile(map[:, :, :, rat_neuron], high_p))
    map = np.clip(map, a_min=cmin, a_max=cmax)

    fig = plot_neuron_2d_3d(
        np.moveaxis(map, 3, 0),
        neuron_idx=rat_neuron,
        axes_names=("X", "Y", "HD"),
        n_slices=4,
        z_spacing=1.0,
        surf_opacity=1.0,
        grid_color="white",
        grid_width=1,
        z_label_degrees=True,
        firing_rate=True,
    )

    fig.write_image(
        current_file_path /
        f"figs/pfeiffer_rat{which_rat + 1}_N{rat_neuron}_rank_{rank_idx}_hd.pdf")
    fig.show()


## Dusz HD Neurons

Duszkiewicz et al 2024

In [ ]:
# generate rate maps from Duszkiewicz data

import os

save_path = current_file_path / f'results/rodents/dusz'
data_path = current_file_path / f'data/rodents/dusz'

dataset_n = [1, 2]

n_loc_bins=20
n_hd_bins=90


In [ ]:
# we only do data from square environments for now
# datasets 1, 2, and 5 are the only ones with square envs
# and 5 only has 2 square out of all of them, so we need to filter

dusz_datasets = {}

from openpyxl import load_workbook

wb = load_workbook(data_path / f"Dataset_metadata.xlsx")

for n in dataset_n:
    sheet = wb[f'Dataset_{n}']

    dataset_dict = {}
    actual = os.listdir(data_path / f'Dataset_{n}')
    for row in sheet.iter_rows(values_only=True):
        if not row[0] or not (row[0][0] == 'A' or row[0][0] == 'B'):
            continue
        key = row[0]
        if key not in actual:
            print("Actual dataset missing:", key)
        values = row[3] # environment shapes and neurons
        dataset_dict[key] = values
    dusz_datasets[n] = dataset_dict


### Plot expected variance for XY vs HD

In [ ]:
from src.dusz_ratemaps import get_ratemaps

ratemap_paths = {}

for n, rats in dusz_datasets.items():
    paths = []
    for rat, env_type in rats.items():
        if env_type != 'Square' or rat == 'B0703-211208': # timestamps are wrong in this one
            continue

        try:
            metadata = get_ratemaps(
                data_path=data_path,
                save_path=save_path,
                dataset=n,
                rat=rat,
                vel_threshold=2,  # matches the stored maps / paper
                n_loc_bins=n_loc_bins,
                n_hd_bins=n_hd_bins,
                floor=5,  # matches the stored maps (min occupancy 4)
            )
            paths.append(metadata['out_npz'])
        except FileNotFoundError as e:
            print(f'Skipping {n}/{rat}: missing input file ({e.filename})')
            continue
    ratemap_paths[n] = paths

In [ ]:
from joblib import Parallel, delayed
from src.explained_variance import *
from tqdm import tqdm

sigma = (1, 1)
max_bins = max(n_loc_bins**2, n_hd_bins)
zoom_factor = (max_bins / n_loc_bins**2, max_bins / n_hd_bins)

def process_neuron_dusz_xyhd(n, fold0_maps, fold1_maps, occ_z, min_occ):
    fold0 = nan_aware_gaussian_filter(fold0_maps[:, :, n], sigma=sigma, circular_axes=(1,))
    fold1 = nan_aware_gaussian_filter(fold1_maps[:, :, n], sigma=sigma, circular_axes=(1,))
    fold0 = masked_zoom(fold0, zoom_factor)
    fold1 = masked_zoom(fold1, zoom_factor)
    ev_xy = 1 - quantify_variance_corr(fold0, fold1, occ_z, axis=0, min_occ=min_occ)[0]
    ev_hd = 1 - quantify_variance_corr(fold0, fold1, occ_z, axis=1, min_occ=min_occ)[0]
    return ev_xy, ev_hd

ev_xy_per_rat = {}
ev_hd_per_rat = {}

for d in dusz_datasets.keys():
    rats = dusz_datasets[d]
    for r_idx, (r, env_type) in enumerate(rats.items()):
        if env_type != 'Square' or r == 'B0703-211208':
            continue

        ratemap_path = save_path / f'Dataset_{d}/{r}/fold_rate_xyhd_{n_loc_bins}x{n_loc_bins}x{n_hd_bins}.npz'
        if not ratemap_path.exists():
            print(f"Skipping {d}/{r}: ratemap not found")
            continue

        maps = np.load(ratemap_path)
        fold0_maps, fold1_maps = maps['fold0_map'], maps['fold1_map']
        occ = maps['occupancy']
        y, x, hd, n_neurons = fold0_maps.shape
        fold0_maps = fold0_maps.reshape(y * x, hd, n_neurons)
        fold1_maps = fold1_maps.reshape(y * x, hd, n_neurons)
        occ = occ.reshape(y * x, hd)
        occ_z = masked_zoom(occ, zoom_factor)
        min_occ = np.percentile(occ_z[occ_z > 0], 10)

        results = Parallel(n_jobs=-1)(
            delayed(process_neuron_dusz_xyhd)(n, fold0_maps, fold1_maps, occ_z, min_occ)
            for n in tqdm(range(n_neurons), desc=f"Dataset {d}/{r}")
        )

        ev_xy_per_rat[(d, r)] = np.array([res[0] for res in results])
        ev_hd_per_rat[(d, r)] = np.array([res[1] for res in results])
        print(f"  Computed EVs for {n_neurons} neurons")

for d in dusz_datasets.keys():
    rats = dusz_datasets[d]
    for r_idx, (r, env_type) in enumerate(rats.items()):
        if env_type != 'Square' or r == 'B0703-211208':
            continue
        if (d, r) not in ev_xy_per_rat:
            continue
        np.save(save_path / f'ev_xy_rat_{r}_all_{n_loc_bins}x{n_loc_bins}x{n_hd_bins}-corr.npy', ev_xy_per_rat[(d, r)])
        np.save(save_path / f'ev_hd_rat_{r}_all_{n_loc_bins}x{n_loc_bins}x{n_hd_bins}-corr.npy', ev_hd_per_rat[(d, r)])
        print(f"Saved EVs for {r}: {len(ev_xy_per_rat[(d, r)])} neurons")


In [ ]:
hd_tuning = np.nanmean(fold0, axis=0)
hd_tuning[np.isnan(hd_tuning)] = 0.

theta = np.linspace(0, 2*np.pi, n_loc_bins*n_loc_bins, endpoint=True)

theta_closed = np.r_[theta, theta[0]]
r_closed = np.r_[hd_tuning, hd_tuning[0]]

fig = plt.figure()
ax = plt.subplot(111, projection='polar')
ax.plot(theta_closed, r_closed, linewidth=2)
ax.fill_between(theta_closed, 0, r_closed, alpha=0.25)

ax.set_theta_zero_location('N')
ax.set_theta_direction(-1)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

max_scatter_points = 10
rng = np.random.default_rng(42)

fig, ax = plt.subplots(figsize=(8, 8))

neuron_counts = []
total_neurons = 0

all_x = np.concatenate(list(ev_xy_per_rat.values())).astype(float)
all_y = np.concatenate(list(ev_hd_per_rat.values())).astype(float)
valid = ~np.isnan(all_x) & ~np.isnan(all_y)
xv, yv = all_x[valid], all_y[valid]

total_plotted = 0
for i, (d, r) in enumerate(ev_xy_per_rat.keys()):
    x_vals = ev_xy_per_rat[(d, r)]
    y_vals = ev_hd_per_rat[(d, r)]
    n_neurons = x_vals.shape[0]
    neuron_counts.append(total_neurons)
    total_neurons += n_neurons

    idx = np.arange(n_neurons)
    rng.shuffle(idx)
    idx = idx[:min(max_scatter_points, n_neurons)]

    ax.scatter(x_vals[idx], y_vals[idx], s=16, alpha=0.8, edgecolors='none', color='C0')
    total_plotted += len(idx)

print(f"Plotted {total_plotted} points (subsampled from {total_neurons} total neurons)")

lo = 0.0 #float(np.nanmin([xv.min(), yv.min()])) if xv.size and yv.size else 0.0
hi = 1.0 #float(np.nanmax([xv.max(), yv.max()])) if xv.size and yv.size else 1.0
pad = 0.02 * (hi - lo if hi > lo else 1.0)
lo -= pad
hi += pad

ax.plot([lo, hi], [lo, hi], '--', lw=1.0, color='red', label='y = x')
ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)
ax.set_aspect('equal', adjustable='box')
ax.set_xlabel('EV (XY)')
ax.set_ylabel('EV (HD)')
ax.legend(loc='best', frameon=False, fontsize=8)

plt.tight_layout()
plt.savefig(current_file_path / f"figs/dusz_xyhd_ev_total_best_fit-corr.pdf")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib import cm
from pathlib import Path
from typing import Mapping, Sequence
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from typing import Tuple, List
import re

def plot_neuron_2d_3d(
    activations: np.ndarray,
    neuron_idx: int,
    axes_names = ("X", "Y", "HD"),
    *,
    n_slices: int = 4,
    z_spacing: float = 1.5,
    cmap: str = "Jet",
    surf_opacity: float = 0.9,
    grid_color: str = "black",
    grid_width: float = 1,
    fig_size = (1800, 600),
    overall_title: str | None = None,
    z_label_degrees: bool = True,
    firing_rate: bool = False,
    show_slice_grid: bool = False,
    hd_polar_quadrants: bool = False
):
    def _col_domains(ncols, column_widths, hspace):
        total_gap = (ncols - 1) * hspace
        avail = 1.0 - total_gap
        norm = sum(column_widths)
        widths = [avail * (w / norm) for w in column_widths]
        domains = []
        start = 0.0
        for i, w in enumerate(widths):
            end = start + w
            domains.append((start, end))
            if i < ncols - 1:
                start = end + hspace
        return domains

    if activations.ndim != 4:
        raise ValueError("`activations` must be 4-D: (N_neurons, Nx, Ny, Nz)")

    data = activations[neuron_idx]
    nx, ny, nz = data.shape

    def idx_to_deg(z_idx: int) -> float:
        return (360.0 / nz) * (z_idx % nz)

    cmin, cmax = float(np.nanmin(data)), float(np.nanmax(data))

    column_widths = [0.35, 0.33, 0.32]
    hspace = 0.07
    x_domains = _col_domains(3, column_widths, hspace)
    xdom2 = x_domains[1]
    xdom3 = x_domains[2]
    ydom = (0.0, 1.0)

    fig = make_subplots(
        rows=1, cols=3,
        column_widths=column_widths,
        specs=[[{"type": "scene"}, {"type": "heatmap"}, {"type": "heatmap"}]],
        horizontal_spacing=hspace,
        subplot_titles=(
            f"3D heatmap ({axes_names[0]}, {axes_names[1]}, {axes_names[2]})",
            f"{axes_names[2]} tuning (avg {axes_names[0]}×{axes_names[1]})",
            f"{axes_names[2]} tuning by {axes_names[0]}×{axes_names[1]} quadrant"
        ),
    )

    X = np.repeat(np.arange(nx), ny * nz)
    Y = np.tile(np.repeat(np.arange(ny), nz), nx)
    Z = np.tile(np.arange(nz), nx * ny)

    fig.add_trace(
        go.Volume(
            x=X, y=Y, z=Z,
            value=data.flatten(),
            opacity=0.10,
            surface_count=25,
            colorscale=cmap,
            cmin=cmin, cmax=cmax,
            showscale=False,
            coloraxis="coloraxis"
        ),
        row=1, col=1
    )
    fig.update_scenes(
        dict(
            xaxis=dict(title=axes_names[0], showticklabels=False),
            yaxis=dict(title=axes_names[1], showticklabels=False),
            zaxis=dict(title=axes_names[2], showticklabels=False),
            aspectmode="cube",
        ),
        row=1, col=1
    )

    fig.update_xaxes(visible=False, row=1, col=2)
    fig.update_yaxes(visible=False, row=1, col=2)
    fig.update_xaxes(visible=False, row=1, col=3)
    fig.update_yaxes(visible=False, row=1, col=3)

    fig.layout["polar"] = dict(
        domain=dict(x=[xdom2[0], xdom2[1]], y=[ydom[0], ydom[1]]),
        radialaxis=dict(showticklabels=False, ticks=""),
        angularaxis=dict(showticklabels=False, ticks="")
    )

    r_full_base = np.nanmean(data, axis=(0, 1))
    theta_full_base = [idx_to_deg(z) for z in range(nz)]

    max_full = float(np.nanmax(r_full_base)) if np.isfinite(np.nanmax(r_full_base)) else float("nan")
    max_full_txt = f"{np.round(max_full, 2):.2f}" if np.isfinite(max_full) else "NA"

    r_full = np.asarray(r_full_base, dtype=float)
    theta_full = list(theta_full_base)
    if len(theta_full) > 0:
        theta_full = theta_full + [theta_full[0] + 360.0]
        r_full = np.concatenate([r_full, [r_full[0]]], axis=0)

    fig.add_trace(
        go.Scatterpolar(
            r=r_full,
            theta=theta_full,
            mode="lines",
            subplot="polar",
            showlegend=False,
            line=dict(color="cornflowerblue"),
            hovertemplate=(
                f"{axes_names[2]}=%{{theta:.0f}}°<br>"
                f"{'Firing rate' if firing_rate else 'Activation'}=%{{r}}"
                "<extra></extra>"
            ),
        )
    )

    fig.add_annotation(
        x=xdom2[0] + 0.02 * (xdom2[1] - xdom2[0]),
        y=ydom[1] - 0.02 * (ydom[1] - ydom[0]),
        xref="paper",
        yref="paper",
        text=f"max {max_full_txt}",
        showarrow=False,
        xanchor="left",
        yanchor="top",
        font=dict(size=12),
    )

    x0, x1 = xdom3
    y0, y1 = ydom
    inner_gap_x = 0.04 * (x1 - x0)
    inner_gap_y = 0.06 * (y1 - y0)

    cell_w = ((x1 - x0) - inner_gap_x) / 2.0
    cell_h = ((y1 - y0) - inner_gap_y) / 2.0

    x_left = (x0, x0 + cell_w)
    x_right = (x0 + cell_w + inner_gap_x, x1)
    y_bottom = (y0, y0 + cell_h)
    y_top = (y0 + cell_h + inner_gap_y, y1)

    polar_domains = [
        (x_left,  y_top,    "polar2", f"{axes_names[0]} low, {axes_names[1]} high"),
        (x_right, y_top,    "polar3", f"{axes_names[0]} high, {axes_names[1]} high"),
        (x_left,  y_bottom, "polar4", f"{axes_names[0]} low, {axes_names[1]} low"),
        (x_right, y_bottom, "polar5", f"{axes_names[0]} high, {axes_names[1]} low"),
    ]

    for (xd, yd, pname, _) in polar_domains:
        fig.layout[pname] = dict(
            domain=dict(x=[xd[0], xd[1]], y=[yd[0], yd[1]]),
            radialaxis=dict(showticklabels=False, ticks=""),
            angularaxis=dict(showticklabels=False, ticks="")
        )

    x_mid = nx // 2
    y_mid = ny // 2
    quad_slices = [
        (slice(0, x_mid), slice(y_mid, ny)),
        (slice(x_mid, nx), slice(y_mid, ny)),
        (slice(0, x_mid), slice(0, y_mid)),
        (slice(x_mid, nx), slice(0, y_mid)),
    ]

    for (xs, ys), (xd, yd, pname, label_txt) in zip(quad_slices, polar_domains):
        r_q_base = np.nanmean(data[xs, ys, :], axis=(0, 1))
        theta_q_base = [idx_to_deg(z) for z in range(nz)]

        max_q = float(np.nanmax(r_q_base)) if np.isfinite(np.nanmax(r_q_base)) else float("nan")
        max_q_txt = f"{np.round(max_q, 2):.2f}" if np.isfinite(max_q) else "NA"

        r_q = np.asarray(r_q_base, dtype=float)
        theta_q = list(theta_q_base)
        if len(theta_q) > 0:
            theta_q = theta_q + [theta_q[0] + 360.0]
            r_q = np.concatenate([r_q, [r_q[0]]], axis=0)

        fig.add_trace(
            go.Scatterpolar(
                r=r_q,
                theta=theta_q,
                mode="lines",
                subplot=pname,
                showlegend=False,
                line=dict(color="cornflowerblue"),
                hovertemplate=(
                    f"{axes_names[2]}=%{{theta:.0f}}°<br>"
                    f"{'Firing rate' if firing_rate else 'Activation'}=%{{r}}"
                    "<extra></extra>"
                ),
            )
        )

        fig.add_annotation(
            x=0.5 * (xd[0] + xd[1]),
            y=yd[1] - 0.01,
            xref="paper",
            yref="paper",
            text=label_txt,
            showarrow=False,
            xanchor="center",
            yanchor="top",
            font=dict(size=12),
        )

        fig.add_annotation(
            x=xd[0] + 0.02 * (xd[1] - xd[0]),
            y=yd[1] - 0.02 * (yd[1] - yd[0]),
            xref="paper",
            yref="paper",
            text=f"max {max_q_txt}",
            showarrow=False,
            xanchor="left",
            yanchor="top",
            font=dict(size=11),
        )

    fig.update_layout(
        showlegend=False,
        coloraxis=dict(
            cmin=cmin, cmax=cmax, colorscale=cmap,
            colorbar=dict(
                title=dict(
                    text="Firing rate<br>(spikes/s)" if firing_rate else "Activation"
                ),
                x=1.02, y=0.5, len=0.9, thickness=20
            )
        ),
        width=fig_size[0], height=fig_size[1],
        margin=dict(l=60, r=100, t=80, b=60),
        title=dict(text=overall_title, x=0.5) if overall_title else None,
    )

    return fig


In [ ]:
#from src.ann.fig import plot_neuron_2d_3d
import plotly.graph_objects as go
from plotly import io as pio

norm_xy = (all_x - lo) / (hi - lo)
norm_hd = (all_y - lo) / (hi - lo)

denom = norm_xy + norm_hd
denom[denom == 0] = np.finfo(float).eps

xy_ratio = norm_xy / denom
hd_ratio = norm_hd / denom

neuron_counts = np.array(neuron_counts)
flat_rat_list = list(ev_xy_per_rat.keys())

most_xy = np.argsort(xy_ratio)[::-1]
most_hd = np.argsort(hd_ratio)[::-1]

rank_idx = 70

global_idx = int(most_hd[rank_idx])
which_rat = np.searchsorted(neuron_counts, global_idx, side='right') - 1
dataset, session = flat_rat_list[which_rat]
rat_neuron = global_idx - neuron_counts[which_rat]
ev = float(hd_ratio[global_idx])

ratemap_path = save_path / f'Dataset_{dataset}/{session}/fold_rate_xyhd_{n_loc_bins}x{n_loc_bins}x{n_hd_bins}.npz'
rate_map = np.load(ratemap_path)['rate_map']

for n in tqdm(range(rate_map.shape[3])):
    rate_map[:, :, :, n] = nan_aware_gaussian_filter(rate_map[:, :, :, n], (1,1,1), circular_axes=(0, 0, 1)) # whatever

# cap outliers
low_p, high_p = 1, 99
cmin = float(np.nanpercentile(rate_map[:, :, :, rat_neuron], low_p))
cmax = float(np.nanpercentile(rate_map[:, :, :, rat_neuron], high_p))
rate_map = np.clip(rate_map, a_min=cmin, a_max=cmax)

fig = plot_neuron_2d_3d(
    np.moveaxis(rate_map, 3, 0),
    neuron_idx=rat_neuron,
    axes_names=("X", "Y", "HD"),
    n_slices=4,
    z_spacing=1.0,
    surf_opacity=1.0,
    grid_color="white",
    grid_width=1,
    z_label_degrees=True,
    firing_rate=True,
    overall_title=f'Dusz Dataset {dataset} Rat {session} Neuron {rat_neuron}, EV_ratio(hd)={ev:.2f}',
    hd_polar_quadrants=True
)

#fig.write_image(current_file_path / f"figs/dusz_rat_{session}_N{rat_neuron}.pdf")
fig.show()


### Plot expected variance for MD vs HD

In [ ]:
n_md_bins = 90
n_hd_bins = 90

In [ ]:
from src.dusz_ratemaps import get_ratemaps

ratemap_paths = {}

for n, rats in dusz_datasets.items():
    paths = []
    for rat, env_type in rats.items():
        if env_type != 'Square' or rat == 'B0703-211208': # timestamps are wrong in this one
            continue

        try:
            metadata = get_ratemaps(
                data_path=data_path,
                save_path=save_path,
                dataset=n,
                rat=rat,
                n_hd_bins=n_hd_bins,
                n_md_bins=n_hd_bins,
                fold=True,
                mode='hdmd',
                md_smooth_window=20,
                vel_threshold=1,
                floor=10
            )
            paths.append(metadata['out_npz'])
        except FileNotFoundError as e:
            print(f'Skipping {n}/{rat}: missing input file ({e.filename})')
            continue
    ratemap_paths[n] = paths

In [ ]:
from joblib import Parallel, delayed
from src.explained_variance import *
from tqdm import tqdm

sigma = (1, 1)
max_bins = max(n_md_bins, n_hd_bins)
zoom_factor = (max_bins / n_md_bins, max_bins / n_hd_bins)

def process_neuron_dusz_hdmd(n, fold0_maps, fold1_maps, occ_z, min_occ):
    fold0 = nan_aware_gaussian_filter(fold0_maps[:, :, n], sigma=sigma, circular_axes=(0, 1))
    fold1 = nan_aware_gaussian_filter(fold1_maps[:, :, n], sigma=sigma, circular_axes=(0, 1))
    fold0 = masked_zoom(fold0, zoom_factor)
    fold1 = masked_zoom(fold1, zoom_factor)
    ev_hd = 1 - quantify_variance_corr(fold0, fold1, occ_z, axis=0, min_occ=min_occ)[0]
    ev_md = 1 - quantify_variance_corr(fold0, fold1, occ_z, axis=1, min_occ=min_occ)[0]
    return ev_md, ev_hd

ev_md_per_rat = {}
ev_hd_per_rat = {}

for d in dusz_datasets.keys():
    rats = dusz_datasets[d]
    for r_idx, (r, env_type) in enumerate(rats.items()):
        if env_type != 'Square' or r == 'B0703-211208':
            continue

        ratemap_path = save_path / f'Dataset_{d}/{r}/fold_rate_hdmd_{n_hd_bins}x{n_md_bins}.npz'
        if not ratemap_path.exists():
            print(f"Skipping {d}/{r}: ratemap not found")
            continue

        maps = np.load(ratemap_path)
        fold0_maps, fold1_maps = maps['fold0_map'], maps['fold1_map']
        occ = maps['occupancy']
        md, hd, n_neurons = fold0_maps.shape
        occ_z = masked_zoom(occ, zoom_factor)
        min_occ = np.percentile(occ_z[occ_z > 0], 10)

        results = Parallel(n_jobs=-1)(
            delayed(process_neuron_dusz_hdmd)(n, fold0_maps, fold1_maps, occ_z, min_occ)
            for n in tqdm(range(n_neurons), desc=f"Dataset {d}/{r}")
        )

        ev_md_per_rat[(d, r)] = np.array([res[0] for res in results])
        ev_hd_per_rat[(d, r)] = np.array([res[1] for res in results])
        print(f"  Computed EVs for {n_neurons} neurons")

for d in dusz_datasets.keys():
    rats = dusz_datasets[d]
    for r_idx, (r, env_type) in enumerate(rats.items()):
        if env_type != 'Square' or r == 'B0703-211208':
            continue
        if (d, r) not in ev_md_per_rat:
            continue
        np.save(save_path / f'ev_md_rat_{r}_all_{n_hd_bins}x{n_md_bins}-corr.npy', ev_md_per_rat[(d, r)])
        np.save(save_path / f'ev_hd_hdmd_rat_{r}_all_{n_hd_bins}x{n_md_bins}-corr.npy', ev_hd_per_rat[(d, r)])
        print(f"Saved EVs for {r}: {len(ev_md_per_rat[(d, r)])} neurons")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

max_scatter_points = 10
rng = np.random.default_rng(42)

fig, ax = plt.subplots(figsize=(8, 8))

neuron_counts = []
total_neurons = 0

all_x = np.concatenate(list(ev_md_per_rat.values())).astype(float)
all_y = np.concatenate(list(ev_hd_per_rat.values())).astype(float)
valid = ~np.isnan(all_x) & ~np.isnan(all_y)
xv, yv = all_x[valid], all_y[valid]

total_plotted = 0
for i, (d, r) in enumerate(ev_md_per_rat.keys()):
    x_vals = ev_md_per_rat[(d, r)]
    y_vals = ev_hd_per_rat[(d, r)]
    n_neurons = x_vals.shape[0]
    neuron_counts.append(total_neurons)
    total_neurons += n_neurons

    idx = np.arange(n_neurons)
    rng.shuffle(idx)
    idx = idx[:min(max_scatter_points, n_neurons)]

    ax.scatter(x_vals[idx], y_vals[idx], s=16, alpha=0.8, edgecolors='none', color='C0')
    total_plotted += len(idx)

print(f"Plotted {total_plotted} points (subsampled from {total_neurons} total neurons)")

lo = 0. #float(np.nanmin([xv.min(), yv.min()])) if xv.size and yv.size else 0.0
hi = 1. #float(np.nanmax([xv.max(), yv.max()])) if xv.size and yv.size else 1.0
pad = 0.02 * (hi - lo if hi > lo else 1.0)
lo -= pad
hi += pad

ax.plot([lo, hi], [lo, hi], '--', lw=1.0, color='red', label='y = x')
ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)
ax.set_aspect('equal', adjustable='box')
ax.set_xlabel('EV (MD)')
ax.set_ylabel('EV (HD)')
ax.legend(loc='best', frameon=False, fontsize=8)

plt.tight_layout()
plt.savefig(current_file_path / f"figs/dusz_hdmd_ev_total_best_fit-corr.pdf")
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.explained_variance import nan_aware_gaussian_filter

# Compute normalized ratios for ranking
norm_md = (all_x - lo) / (hi - lo)
norm_hd = (all_y - lo) / (hi - lo)

denom = norm_md + norm_hd
denom[denom == 0] = np.finfo(float).eps

md_ratio = norm_md / denom
hd_ratio = norm_hd / denom

neuron_counts = np.array(neuron_counts)
flat_rat_list = list(ev_md_per_rat.keys())

most_md = np.argsort(md_ratio)[::-1]
most_hd = np.argsort(hd_ratio)[::-1]

# Select 2 neurons to plot: one high-MD, one high-HD
neurons_to_plot = [
    #{"rank_idx": 20, "source": most_md, "label": "High MD"},
    #{"rank_idx": 1, "source": most_hd, "label": "High HD"},
    #{"rank_idx": 2, "source": most_hd, "label": "High HD"},
    #{"rank_idx": 3, "source": most_hd, "label": "High HD"},
]

for neuron_info in neurons_to_plot:
    rank_idx = neuron_info["rank_idx"]
    source = neuron_info["source"]
    label = neuron_info["label"]

    global_idx = int(source[rank_idx])
    which_rat = np.searchsorted(neuron_counts, global_idx, side='right') - 1
    dataset, session = flat_rat_list[which_rat]
    rat_neuron = global_idx - neuron_counts[which_rat]
    ev_md = float(md_ratio[global_idx])
    ev_hd = float(hd_ratio[global_idx])

    print(f"\n=== {label} neuron (rank {rank_idx}) ===")
    print(f"  Dataset: {dataset}, Session: {session}, Neuron: {rat_neuron}")
    print(f"  MD ratio: {ev_md:.3f}, HD ratio: {ev_hd:.3f}")

    # Load rate map
    ratemap_path = save_path / f'Dataset_{dataset}/{session}/fold_rate_hdmd_{n_hd_bins}x{n_md_bins}.npz'
    data = np.load(ratemap_path)
    rate_map = data['rate_map']
    occ = data['occupancy']

    # Smooth the rate map for this neuron
    smoothed_map = nan_aware_gaussian_filter(rate_map[:, :, rat_neuron], (1, 1), circular_axes=(1, 1))

    # Compute marginal tuning curves
    spikes = occ * smoothed_map

    # MD curve (marginalize over HD, axis=0)
    spikes_slice_md = np.nansum(spikes, axis=0)
    time_md = np.nansum(occ, axis=0)
    md_curve = spikes_slice_md / time_md

    # HD curve (marginalize over MD, axis=1)
    spikes_slice_hd = np.nansum(spikes, axis=1)
    time_hd = np.nansum(occ, axis=1)
    hd_curve = spikes_slice_hd / time_hd

    # Prepare polar coordinates
    theta_hd = np.linspace(0, 2*np.pi, hd_curve.size, endpoint=False)
    theta_md = np.linspace(0, 2*np.pi, md_curve.size, endpoint=False)

    theta_hd_closed = np.r_[theta_hd, theta_hd[0]]
    hd_closed = np.r_[hd_curve, hd_curve[0]]

    theta_md_closed = np.r_[theta_md, theta_md[0]]
    md_closed = np.r_[md_curve, md_curve[0]]

    # Create figure with 3 subplots: HD polar, MD polar, joint heatmap
    fig = plt.figure(figsize=(15, 4), constrained_layout=True)

    # HD polar plot
    ax1 = fig.add_subplot(1, 3, 1, projection='polar')
    ax1.plot(theta_hd_closed, hd_closed, linewidth=2)
    ax1.fill_between(theta_hd_closed, 0, hd_closed, alpha=0.3)
    ax1.set_theta_zero_location("N")
    ax1.set_theta_direction(-1)
    ax1.set_title("HD Tuning", fontsize=12)

    # MD polar plot
    ax2 = fig.add_subplot(1, 3, 2, projection='polar')
    ax2.plot(theta_md_closed, md_closed, linewidth=2, color='C1')
    ax2.fill_between(theta_md_closed, 0, md_closed, alpha=0.3, color='C1')
    ax2.set_theta_zero_location("N")
    ax2.set_theta_direction(-1)
    ax2.set_title("MD Tuning", fontsize=12)

    # Joint HD/MD heatmap
    ax3 = fig.add_subplot(1, 3, 3)
    im = ax3.imshow(smoothed_map.T, origin='lower', aspect='equal', cmap='viridis')
    ax3.set_xlabel("HD bin")
    ax3.set_ylabel("MD bin")
    ax3.set_title("Joint HD×MD Tuning", fontsize=12)
    plt.colorbar(im, ax=ax3, label="Firing rate")

    fig.suptitle(f"{label} Neuron - D{dataset}/{session} N{rat_neuron} (rank {rank_idx})", fontsize=14)

    fig.savefig(current_file_path / f'figs/dusz_hdmd_D{dataset}_{session}_N{rat_neuron}_{label.replace(" \"", "_\"")}-corr.pdf')
    plt.show()

In [ ]:
# Plot joint HD/MD occupancy distribution from trajectory data
import matplotlib.pyplot as plt
import numpy as np

# Plot all individual sessions
total_occ = None
for (d, r) in ev_md_per_rat.keys():
    occ_path = save_path / f'Dataset_{d}/{r}/fold_rate_hdmd_{n_hd_bins}x{n_md_bins}.npz'
    occ = np.load(occ_path)['occupancy']

    # Accumulate for aggregate
    if total_occ is None:
        total_occ = occ.copy()
    else:
        total_occ += occ

    # Plot individual session
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(occ.T, origin='lower', aspect='equal', cmap='viridis',
                   extent=[0, 360, 0, 360])
    ax.set_xlabel('Head Direction (°)')
    ax.set_ylabel('Movement Direction (°)')
    ax.set_title(f'Joint HD/MD Occupancy\n(Dataset {d}, Session {r})')
    plt.colorbar(im, ax=ax, label='Time (s)')
    plt.tight_layout()
    plt.savefig(current_file_path / f'figs/hdmd_occupancy_D{d}_{r}-corr.pdf')
    plt.show()

# Plot aggregate across all sessions
fig, ax = plt.subplots(figsize=(6, 5))
total_occ_norm = total_occ / np.nansum(total_occ)

im = ax.imshow(total_occ_norm.T, origin='lower', aspect='equal', cmap='viridis',
               extent=[0, 360, 0, 360])
ax.set_xlabel('Head Direction (°)')
ax.set_ylabel('Movement Direction (°)')
ax.set_title('Joint HD/MD Distribution (All Sessions)')
plt.colorbar(im, ax=ax, label='P(HD, MD)')
plt.tight_layout()
plt.savefig(current_file_path / f'figs/hdmd_occupancy_all-corr.pdf')
plt.show()

print(f"Plotted {len(ev_md_per_rat)} individual sessions + 1 aggregate")